# Scikit learn gaussian process regression for Air Quality Index

a notebook to optmize a gaussian process using Scikit learns implementation, for comparison againt MuyGPs

In [9]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
import time

## Import/Setup data

In [10]:
BASE_DIR = os.path.abspath("..") + "\\"
aqi_data = np.genfromtxt(BASE_DIR + "data\\biggest_day_aqi_coords_scaled_noAH.csv", delimiter=',', skip_header=1)
#print(aqi_data[:10])

#randomly split the data into features and responces, and those in turn into training and testing paritions.
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]
aqi_x_train, aqi_x_test, aqi_y_train, aqi_y_test = train_test_split(aqi_x, aqi_y, test_size=0.1, random_state=23)

#use the mean of the training points to provide basic de-meaning of the points.
aqi_y_mean = aqi_y_train.mean()
aqi_y_train_centered = aqi_y_train - aqi_y_mean

## Selecting a kernel

In [11]:
#kernel = 1.0 * RBF(length_scale=0.01)
kernel = 1.0 * Matern(length_scale=1.432, nu=1e-6)

## Creating the GP model

In [12]:
scikit_gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=40)

## Training the model

In [13]:
optimize_start = time.time()

scikit_gp.fit(aqi_x_train, aqi_y_train)

optimize_end = time.time()
optimize_length = optimize_end - optimize_start
print(f"Gaussian Process optimization took {optimize_length} seconds")

Gaussian Process optimization took 115.28478837013245 seconds


## Making Predictions

In [14]:
scikit_predicted_centered, scikit_sigma = scikit_gp.predict(aqi_x_test, return_std=True)
scikit_predicted = scikit_predicted_centered + aqi_y_mean
print(scikit_predicted[:10])
print(scikit_sigma[:10])

[82.96566322 66.78957876 93.02745973 66.52552037 92.90216123 57.90052588
 53.89526812 86.75355521 96.28101939 84.47553516]
[24.5518569  40.26116129 31.03328614 35.56830595 36.4468651  33.60726648
 21.4993868  33.06190997 19.96540197  7.62397093]


In [15]:
# calculate values that equate to the ones used to rank muyGPs. Scikit returns the variances already square rooted.
confidence_intervals = scikit_sigma * 1.96
mean_confidence_interval = confidence_intervals.mean()

# Evaluate against the ORIGINAL (un-centered) test responses.
coverage = np.count_nonzero(
    np.abs(aqi_y_test - scikit_predicted) < confidence_intervals
) / aqi_y_test.size

rmse = np.sqrt(np.mean((aqi_y_test - scikit_predicted) ** 2))
print(f"RMSE: {rmse:.4f}")
print(f"Coverage: {coverage:.4f}")
print(f"Mean Confidence Interval: {mean_confidence_interval:.4f}")

RMSE: 45.3353
Coverage: 0.6422
Mean Confidence Interval: 51.6661
